# EXP 00 — Data Audit, Match-Level Canonicalization, Offline AW-MAE Evaluator, Temporal Validation, dan Fondasi Pipeline Kompetisi

Notebook ini adalah **eksperimen fondasi** untuk kompetisi prediksi skor pertandingan sepak bola internasional. Fokus utamanya bukan membangun model besar, tetapi memastikan seluruh infrastruktur eksperimen sudah benar sejak awal.

Tujuan utama notebook ini:

- memvalidasi file input kompetisi,
- memahami struktur data train, test, sample submission, dan metadata,
- membuktikan bahwa **1 pertandingan direpresentasikan oleh 2 row**,
- membangun representasi **match-level canonical** yang deterministik dan reversible,
- mengimplementasikan **offline evaluator AW-MAE** sesuai definisi kompetisi,
- menyiapkan **temporal validation split** yang leakage-safe,
- mengaudit gap fitur train vs test sebagai dasar eksperimen berikutnya.

Karena metrik kompetisi adalah **AW-MAE**, notebook ini juga menyiapkan evaluator custom yang menghukum kesalahan outcome, exact score, dan goal difference. Artinya, fondasi evaluasi dan representasi data harus benar sebelum masuk ke eksperimen modeling selanjutnya.

## 01. Setup dan Konfigurasi

Section ini menyiapkan library, seed, path relatif, serta helper dasar yang dipakai di seluruh notebook. Karena notebook disimpan di folder `notebook/`, maka seluruh akses data dilakukan melalui path relatif `../data/...`.

Seed diset sejak awal agar eksperimen lebih konsisten dan mudah direproduksi. Meskipun EXP 00 belum berfokus pada model kompleks, helper seed umum tetap disiapkan karena akan berguna untuk eksperimen lanjutan.

In [ ]:

import os
import sys
import json
import math
import random
import warnings
import textwrap
from pathlib import Path
from collections import Counter
from typing import Any, Dict, List, Tuple
from IPython.display import display

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

try:
    import seaborn as sns
except Exception:
    sns = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

warnings.filterwarnings("ignore")

SEED = 42

TRAIN_PATH = Path("../data/train.csv")
TEST_PATH = Path("../data/test.csv")
SAMPLE_SUB_PATH = Path("../data/sample_submission.csv")
META_PATH = Path("../data/metadata.txt")

FILE_PATHS = {
    "train": TRAIN_PATH,
    "test": TEST_PATH,
    "sample_submission": SAMPLE_SUB_PATH,
    "metadata": META_PATH,
}

NOTEBOOK_WORKDIR = Path.cwd()
print(f"Current working directory: {NOTEBOOK_WORKDIR.resolve()}")
print(f"Python version: {sys.version.split()[0]}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

In [ ]:

def seed_everything(seed: int = 42) -> None:
    """Set seed for basic reproducibility."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


def validate_input_files(file_paths: Dict[str, Path]) -> None:
    """Validate that all required input files exist."""
    missing_files = []
    for name, path in file_paths.items():
        if not Path(path).exists():
            missing_files.append((name, str(path)))

    if missing_files:
        missing_msg = "\n".join([f"- {name}: {path}" for name, path in missing_files])
        raise FileNotFoundError(
            "Beberapa file input tidak ditemukan. Pastikan struktur project benar dan notebook dijalankan dari folder "
            "`notebook/`.\n" + missing_msg
        )


def format_file_size_mb(path: Path) -> float:
    return round(path.stat().st_size / (1024 ** 2), 4)


def safe_parse_datetime(series: pd.Series) -> pd.Series:
    """Safely parse a date-like series into pandas datetime."""
    parsed = pd.to_datetime(series, errors="coerce")
    return parsed


def load_csv_safe(path: Path, date_col: str = "date") -> pd.DataFrame:
    """Load CSV defensively and parse date column when available."""
    if not Path(path).exists():
        raise FileNotFoundError(f"CSV file tidak ditemukan: {path}")

    df = pd.read_csv(path, low_memory=False)

    if date_col in df.columns:
        original_non_null = df[date_col].notna().sum()
        df[date_col] = safe_parse_datetime(df[date_col])
        parsed_non_null = df[date_col].notna().sum()
        if original_non_null > 0 and parsed_non_null < original_non_null:
            print(
                f"[WARN] Kolom `{date_col}` pada {path.name} memiliki "
                f"{original_non_null - parsed_non_null} nilai yang gagal diparse menjadi datetime."
            )

    return df


def summarize_dataframe(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Return a compact summary table for a dataframe."""
    summary = {
        "dataset": name,
        "n_rows": len(df),
        "n_cols": df.shape[1],
        "n_missing_cells": int(df.isna().sum().sum()),
        "n_duplicate_rows": int(df.duplicated().sum()),
    }

    if "date" in df.columns:
        summary["date_min"] = df["date"].min()
        summary["date_max"] = df["date"].max()
        summary["n_date_na"] = int(df["date"].isna().sum())

    return pd.DataFrame([summary])


def safe_scalar_equal(a: Any, b: Any) -> bool:
    """Robust scalar comparison that treats NaN == NaN as True."""
    if pd.isna(a) and pd.isna(b):
        return True
    return a == b


seed_everything(SEED)
print(f"Seed set to {SEED}")

## 02. Validasi File Input

Tahap ini memastikan bahwa environment dan struktur project sudah benar **sebelum** data dibaca lebih jauh. Jika file tidak ditemukan sejak awal, notebook akan gagal dengan pesan yang jelas sehingga debugging menjadi lebih mudah.

In [ ]:

validate_input_files(FILE_PATHS)
print("Semua file input utama ditemukan.")

In [ ]:

file_info_rows = []
for name, path in FILE_PATHS.items():
    file_info_rows.append(
        {
            "file_key": name,
            "path": str(path),
            "exists": path.exists(),
            "size_mb": format_file_size_mb(path),
        }
    )

file_info_df = pd.DataFrame(file_info_rows).sort_values("file_key").reset_index(drop=True)
file_info_df

## 03. Load Data dan Metadata

Setelah file tervalidasi, data utama kompetisi dibaca dengan parsing yang aman. Kolom `date` diparse menjadi `datetime` agar siap dipakai untuk audit temporal dan holdout berbasis waktu.

Metadata juga dibaca sebagai raw text agar konteks kompetisi tetap terdokumentasi langsung di notebook.

In [ ]:

train_df = load_csv_safe(TRAIN_PATH, date_col="date")
test_df = load_csv_safe(TEST_PATH, date_col="date")
sample_sub_df = load_csv_safe(SAMPLE_SUB_PATH, date_col="date")

print("Summary ringkas dataframe:")
summary_df = pd.concat(
    [
        summarize_dataframe(train_df, "train"),
        summarize_dataframe(test_df, "test"),
        summarize_dataframe(sample_sub_df, "sample_submission"),
    ],
    ignore_index=True,
)
summary_df

In [ ]:

metadata_raw = META_PATH.read_text(encoding="utf-8", errors="ignore")
metadata_preview = metadata_raw if len(metadata_raw) <= 4000 else metadata_raw[:4000] + "\n\n...[truncated]..."
print(metadata_preview)

In [ ]:

shape_info_df = pd.DataFrame(
    {
        "dataset": ["train", "test", "sample_submission"],
        "shape": [train_df.shape, test_df.shape, sample_sub_df.shape],
        "n_columns": [train_df.shape[1], test_df.shape[1], sample_sub_df.shape[1]],
    }
)
shape_info_df

In [ ]:

print("Head of train:")
display(train_df.head())

print("\nHead of test:")
display(test_df.head())

print("\nHead of sample submission:")
display(sample_sub_df.head())

Ringkasan awal pada tahap loading:

- train, test, dan sample submission berhasil dibaca dari folder `../data/`,
- sample submission perlu konsisten dengan row-level test,
- metadata berhasil dibaca sebagai referensi tambahan untuk memahami aturan kompetisi,
- kolom `date` sudah dipaksa ke format datetime agar siap dipakai dalam audit dan split temporal.

## 04. Audit Struktur Kolom

Section ini memetakan overlap dan gap antara train vs test. Ini sangat penting karena kompetisi memiliki **feature mismatch**: banyak fitur performa historis tersedia di train tetapi tidak tersedia di test.

Konsekuensinya, eksperimen berikutnya harus berhati-hati agar tidak membangun pipeline yang bergantung penuh pada fitur train-only.

In [ ]:

train_cols = set(train_df.columns)
test_cols = set(test_df.columns)

train_only_cols = sorted(train_cols - test_cols)
test_only_cols = sorted(test_cols - train_cols)
shared_cols = sorted(train_cols & test_cols)

print(f"Jumlah kolom overlap     : {len(shared_cols)}")
print(f"Jumlah kolom train-only  : {len(train_only_cols)}")
print(f"Jumlah kolom test-only   : {len(test_only_cols)}")

print("\nKolom hanya di train:")
print(train_only_cols)

print("\nKolom hanya di test:")
print(test_only_cols)

print("\nContoh kolom overlap:")
print(shared_cols[:50])

In [ ]:

all_cols = sorted(train_cols | test_cols)
column_audit_df = pd.DataFrame(
    {
        "column_name": all_cols,
        "in_train": [col in train_df.columns for col in all_cols],
        "in_test": [col in test_df.columns for col in all_cols],
        "dtype_train": [str(train_df[col].dtype) if col in train_df.columns else None for col in all_cols],
        "dtype_test": [str(test_df[col].dtype) if col in test_df.columns else None for col in all_cols],
    }
)
column_audit_df.head(100)

In [ ]:

target_cols = ["team_goals", "opp_goals"]
target_audit_df = pd.DataFrame(
    {
        "target_col": target_cols,
        "exists_in_train": [col in train_df.columns for col in target_cols],
        "exists_in_test": [col in test_df.columns for col in target_cols],
    }
)
target_audit_df

Dari audit struktur kolom ini, hal yang perlu diingat adalah:

- **target** seharusnya hanya muncul di train,
- adanya fitur train-only berarti validasi offline bisa menjadi terlalu optimistis jika kita membangun pipeline yang tidak feasible saat inference di test,
- karena itu, eksperimen awal yang sehat sebaiknya dimulai dari **shared features** terlebih dahulu, lalu train-only features dipertimbangkan secara hati-hati pada eksperimen lanjutan.

## 05. Audit Tipe Data dan Missing Values

Section ini berfokus pada kualitas dasar data: tipe kolom, missing values, dan sentinel value yang berpotensi perlu dibersihkan pada eksperimen berikutnya.

In [ ]:

train_dtype_df = train_df.dtypes.rename("dtype").reset_index().rename(columns={"index": "column"})
test_dtype_df = test_df.dtypes.rename("dtype").reset_index().rename(columns={"index": "column"})

print("Tipe data train:")
display(train_dtype_df.head(100))

print("\nTipe data test:")
display(test_dtype_df.head(100))

In [ ]:

train_missing_df = (
    train_df.isna()
    .sum()
    .rename("n_missing")
    .reset_index()
    .rename(columns={"index": "column"})
)
train_missing_df["pct_missing"] = train_missing_df["n_missing"] / len(train_df) * 100
train_missing_df = train_missing_df.sort_values(["n_missing", "column"], ascending=[False, True]).reset_index(drop=True)
train_missing_df.head(50)

In [ ]:

test_missing_df = (
    test_df.isna()
    .sum()
    .rename("n_missing")
    .reset_index()
    .rename(columns={"index": "column"})
)
test_missing_df["pct_missing"] = test_missing_df["n_missing"] / len(test_df) * 100
test_missing_df = test_missing_df.sort_values(["n_missing", "column"], ascending=[False, True]).reset_index(drop=True)
test_missing_df.head(50)

In [ ]:

sentinel_rows = []
for df_name, df in [("train", train_df), ("test", test_df)]:
    if "altitude_venue" in df.columns:
        count_sentinel = int((df["altitude_venue"] == -9999).fillna(False).sum())
        sentinel_rows.append(
            {"dataset": df_name, "column": "altitude_venue", "sentinel_value": -9999, "count": count_sentinel}
        )

if sentinel_rows:
    sentinel_audit_df = pd.DataFrame(sentinel_rows)
else:
    sentinel_audit_df = pd.DataFrame(
        [{"dataset": "N/A", "column": "altitude_venue", "sentinel_value": -9999, "count": 0}]
    )
sentinel_audit_df

In [ ]:

print("Kolom train dengan missing tertinggi:")
display(train_missing_df.head(20))

print("\nKolom test dengan missing tertinggi:")
display(test_missing_df.head(20))

Insight penting dari audit kualitas data:

- beberapa fitur kemungkinan memiliki missing yang cukup besar dan perlu dipetakan sebelum modeling,
- fitur venue/geografis/sosio-ekonomi sering kali menjadi kandidat yang rawan missing,
- sentinel value seperti `-9999` pada `altitude_venue` **belum diubah** di EXP 00; di sini kita hanya mengauditnya agar pembersihan data bisa dilakukan secara sadar pada eksperimen berikutnya.

## 06. Audit Uniqueness, Match Pairing, dan Konsistensi Dua Baris per Pertandingan

Tujuan section ini adalah memastikan bahwa **1 pertandingan benar-benar direpresentasikan oleh 2 row** dan bahwa pasangan row tersebut konsisten satu sama lain.

In [ ]:

def audit_match_pairing(df: pd.DataFrame, is_train: bool = True) -> Dict[str, Any]:
    required_cols = {"match_id", "team", "opponent"}
    missing_required = required_cols - set(df.columns)
    if missing_required:
        raise KeyError(f"Kolom wajib untuk audit pairing tidak ditemukan: {sorted(missing_required)}")

    group_sizes = df.groupby("match_id").size().sort_values(ascending=False)
    invalid_match_ids = group_sizes[group_sizes != 2].index.tolist()

    valid_match_ids = group_sizes[group_sizes == 2].index
    valid_df = df[df["match_id"].isin(valid_match_ids)].copy()

    base_equal_cols = [col for col in ["date", "tournament", "venue_country", "neutral", "gender"] if col in df.columns]
    audit_counter = Counter()
    issue_examples = []

    for match_id, group in valid_df.groupby("match_id", sort=False):
        if "Id" in group.columns:
            group = group.sort_values(["Id"], kind="stable").reset_index(drop=True)
        else:
            group = group.sort_values(["team", "opponent"], kind="stable").reset_index(drop=True)

        row_a = group.iloc[0]
        row_b = group.iloc[1]

        team_mirror_ok = (
            safe_scalar_equal(row_a["team"], row_b["opponent"])
            and safe_scalar_equal(row_a["opponent"], row_b["team"])
        )
        audit_counter["team_opponent_mirror_ok" if team_mirror_ok else "team_opponent_mirror_fail"] += 1

        if not team_mirror_ok and len(issue_examples) < 10:
            issue_examples.append(
                {
                    "match_id": match_id,
                    "issue": "team/opponent mirror mismatch",
                    "row_a_team": row_a["team"],
                    "row_a_opponent": row_a["opponent"],
                    "row_b_team": row_b["team"],
                    "row_b_opponent": row_b["opponent"],
                }
            )

        for col in base_equal_cols:
            ok = safe_scalar_equal(row_a[col], row_b[col])
            audit_counter[f"{col}_consistent" if ok else f"{col}_inconsistent"] += 1

        if is_train and {"team_goals", "opp_goals"}.issubset(group.columns):
            goals_mirror_ok = (
                safe_scalar_equal(row_a["team_goals"], row_b["opp_goals"])
                and safe_scalar_equal(row_a["opp_goals"], row_b["team_goals"])
            )
            audit_counter["goal_mirror_ok" if goals_mirror_ok else "goal_mirror_fail"] += 1

            if not goals_mirror_ok and len(issue_examples) < 10:
                issue_examples.append(
                    {
                        "match_id": match_id,
                        "issue": "goal mirror mismatch",
                        "row_a_team_goals": row_a["team_goals"],
                        "row_a_opp_goals": row_a["opp_goals"],
                        "row_b_team_goals": row_b["team_goals"],
                        "row_b_opp_goals": row_b["opp_goals"],
                    }
                )

        if "is_home" in group.columns:
            home_vals = pd.to_numeric(group["is_home"], errors="coerce").tolist()
            if pd.notna(home_vals[0]) and pd.notna(home_vals[1]):
                if abs((home_vals[0] + home_vals[1]) - 1) < 1e-12:
                    audit_counter["is_home_complement"] += 1
                elif home_vals[0] == home_vals[1]:
                    audit_counter["is_home_same"] += 1
                else:
                    audit_counter["is_home_other_pattern"] += 1
            else:
                audit_counter["is_home_missing_pattern"] += 1

    result = {
        "n_rows": int(len(df)),
        "n_unique_match_id": int(df["match_id"].nunique()),
        "group_size_distribution": group_sizes.value_counts().sort_index().to_dict(),
        "n_invalid_match_ids_not_equal_2": int(len(invalid_match_ids)),
        "invalid_match_ids_sample": invalid_match_ids[:10],
        "audit_counter": dict(audit_counter),
        "issue_examples": issue_examples,
    }
    return result

In [ ]:

print(f"Id unique di train: {'Id' in train_df.columns and train_df['Id'].is_unique}")
print(f"Id unique di test : {'Id' in test_df.columns and test_df['Id'].is_unique}")

print(f"match_id unique di train: {train_df['match_id'].is_unique if 'match_id' in train_df.columns else 'N/A'}")
print(f"match_id unique di test : {test_df['match_id'].is_unique if 'match_id' in test_df.columns else 'N/A'}")

train_group_size_dist = train_df.groupby("match_id").size().value_counts().sort_index() if "match_id" in train_df.columns else pd.Series(dtype=int)
test_group_size_dist = test_df.groupby("match_id").size().value_counts().sort_index() if "match_id" in test_df.columns else pd.Series(dtype=int)

print("\nDistribusi jumlah row per match_id pada train:")
display(train_group_size_dist.rename_axis("rows_per_match").reset_index(name="n_match_ids"))

print("\nDistribusi jumlah row per match_id pada test:")
display(test_group_size_dist.rename_axis("rows_per_match").reset_index(name="n_match_ids"))

In [ ]:

train_non_pair_count = int((train_df.groupby("match_id").size() != 2).sum())
test_non_pair_count = int((test_df.groupby("match_id").size() != 2).sum())

print(f"Jumlah match_id train yang tidak punya tepat 2 row: {train_non_pair_count}")
print(f"Jumlah match_id test yang tidak punya tepat 2 row : {test_non_pair_count}")

In [ ]:

example_match_ids = train_df["match_id"].drop_duplicates().head(3).tolist() if "match_id" in train_df.columns else []
for match_id in example_match_ids:
    print(f"\nContoh pasangan row untuk match_id = {match_id}")
    display(train_df.loc[train_df["match_id"] == match_id].copy())

In [ ]:

train_pair_audit = audit_match_pairing(train_df, is_train=True)
test_pair_audit = audit_match_pairing(test_df, is_train=False)

print("Ringkasan audit pairing train:")
display(pd.DataFrame([train_pair_audit]).T.rename(columns={0: "value"}))

print("\nRingkasan audit pairing test:")
display(pd.DataFrame([test_pair_audit]).T.rename(columns={0: "value"}))

print("\nContoh issue train (jika ada):")
display(pd.DataFrame(train_pair_audit["issue_examples"]))

print("\nContoh issue test (jika ada):")
display(pd.DataFrame(test_pair_audit["issue_examples"]))

Hasil dari audit pairing ini menentukan apakah pipeline bisa dipindahkan dengan aman dari representasi row-level ke **match-level**. Jika satu pertandingan memang konsisten direpresentasikan oleh 2 row, maka kita bisa:

- melakukan split pada level pertandingan,
- mengevaluasi loss per pertandingan,
- dan melakukan decoding skor pasangan secara lebih konsisten.

Itulah alasan mengapa canonicalization match-level menjadi langkah penting berikutnya.

## 07. Match-Level Canonicalization

Section ini adalah inti dari EXP 00. Data row-level dua-baris-per-match diubah menjadi **satu row per pertandingan** dengan representasi yang stabil, deterministik, dan bisa dibalik lagi ke format row-level submission.

Aturan deterministik yang dipakai di notebook ini:

- jika kolom `Id` tersedia, pasangan row diurutkan berdasarkan `Id` naik,
- jika `Id` tidak tersedia, fallback ke urutan alfabetis `team`, lalu `opponent`.

Dengan aturan ini, sisi canonical `team_a` dan `team_b` akan selalu terbentuk secara konsisten.

In [ ]:

def build_match_level(df: pd.DataFrame, is_train: bool) -> pd.DataFrame:
    required_cols = {"match_id", "team", "opponent"}
    missing_required = required_cols - set(df.columns)
    if missing_required:
        raise KeyError(f"Kolom wajib untuk canonicalization tidak ditemukan: {sorted(missing_required)}")

    group_sizes = df.groupby("match_id").size()
    bad_match_ids = group_sizes[group_sizes != 2].index.tolist()
    if bad_match_ids:
        raise ValueError(
            "Canonicalization membutuhkan tepat 2 row per match_id. "
            f"Ditemukan {len(bad_match_ids)} match_id bermasalah. Contoh: {bad_match_ids[:10]}"
        )

    known_match_level_cols = {"match_id", "date", "gender", "tournament", "venue_country", "neutral"}
    detected_match_level_cols = set(col for col in df.columns if col.startswith("venue_") or col.endswith("_venue"))
    match_level_cols = [col for col in df.columns if col in known_match_level_cols or col in detected_match_level_cols]

    skip_cols = set(match_level_cols) | {"Id", "team", "opponent"}
    if is_train:
        skip_cols |= {"team_goals", "opp_goals"}
    else:
        skip_cols |= {"team_goals", "opp_goals"}

    records: List[Dict[str, Any]] = []

    for match_id, group in df.groupby("match_id", sort=False):
        if "Id" in group.columns:
            group = group.sort_values(["Id"], kind="stable").reset_index(drop=True)
        else:
            group = group.sort_values(["team", "opponent"], kind="stable").reset_index(drop=True)

        row_a = group.iloc[0]
        row_b = group.iloc[1]

        record: Dict[str, Any] = {"match_id": match_id}

        for col in match_level_cols:
            if col == "match_id":
                continue
            val_a = row_a[col] if col in group.columns else np.nan
            val_b = row_b[col] if col in group.columns else np.nan

            if safe_scalar_equal(val_a, val_b):
                record[col] = val_a
            else:
                # Ambil nilai dari row_a sebagai fallback, tetapi tetap mempertahankan proses agar notebook tidak crash.
                record[col] = val_a

        record["team_a"] = row_a["team"]
        record["team_b"] = row_b["team"]

        if "Id" in group.columns:
            record["row_id_a"] = row_a["Id"]
            record["row_id_b"] = row_b["Id"]

        if "is_home" in group.columns:
            record["team_a_is_home"] = row_a["is_home"]
            record["team_b_is_home"] = row_b["is_home"]

        if is_train and {"team_goals", "opp_goals"}.issubset(group.columns):
            record["team_a_goals"] = row_a["team_goals"]
            record["team_b_goals"] = row_b["team_goals"]

        for col in df.columns:
            if col in skip_cols or col == "is_home":
                continue

            # Abaikan kolom opponent_* / opp_* karena secara konsep akan terwakili oleh row pasangan.
            if col.startswith("opponent_") or col.startswith("opp_"):
                continue

            if col.startswith("team_"):
                suffix = col[len("team_"):]
                record[f"team_a_{suffix}"] = row_a[col]
                record[f"team_b_{suffix}"] = row_b[col]
            else:
                # Plain row-level features diperlakukan sebagai fitur sisi tim saat ini.
                record[f"team_a_{col}"] = row_a[col]
                record[f"team_b_{col}"] = row_b[col]

        records.append(record)

    match_df = pd.DataFrame(records)

    preferred_front_cols = [
        "match_id",
        "date",
        "gender",
        "tournament",
        "venue_country",
        "neutral",
        "row_id_a",
        "row_id_b",
        "team_a",
        "team_b",
        "team_a_is_home",
        "team_b_is_home",
        "team_a_goals",
        "team_b_goals",
    ]
    existing_front_cols = [col for col in preferred_front_cols if col in match_df.columns]
    remaining_cols = [col for col in match_df.columns if col not in existing_front_cols]
    match_df = match_df[existing_front_cols + remaining_cols].copy()

    if "date" in match_df.columns:
        match_df["date"] = safe_parse_datetime(match_df["date"])

    return match_df

In [ ]:

train_match = build_match_level(train_df, is_train=True)
test_match = build_match_level(test_df, is_train=False)

print("Shape train_match:", train_match.shape)
print("Shape test_match :", test_match.shape)

display(train_match.head())
display(test_match.head())

In [ ]:

verification_df = pd.DataFrame(
    {
        "dataset": ["train_match", "test_match"],
        "n_rows_match_level": [len(train_match), len(test_match)],
        "n_unique_match_id_row_level": [
            train_df["match_id"].nunique(),
            test_df["match_id"].nunique(),
        ],
        "is_equal": [
            len(train_match) == train_df["match_id"].nunique(),
            len(test_match) == test_df["match_id"].nunique(),
        ],
    }
)
verification_df

Representasi match-level ini penting karena:

- satu pertandingan sekarang punya **satu identitas yang jelas**,
- split temporal bisa dilakukan di level pertandingan, bukan row,
- evaluasi loss menjadi lebih natural pada pasangan skor,
- eksperimen lanjutan bisa membangun model joint score yang lebih konsisten,
- dan prediksi tetap bisa dikembalikan ke format row-level submission melalui helper reverse mapping.

## 08. Reverse Mapping ke Format Submission

Walaupun eksperimen dan evaluasi lebih nyaman dikerjakan di level pertandingan, format submit kompetisi tetap berada di **row-level**. Karena itu, kita perlu helper yang mengubah prediksi match-level kembali menjadi bentuk:

- `Id`
- `team_goals`
- `opp_goals`

In [ ]:

def match_predictions_to_submission(
    test_row_df: pd.DataFrame,
    pred_match_df: pd.DataFrame,
    canonical_team_a_col: str = "team_a",
    canonical_team_b_col: str = "team_b",
    pred_a_col: str = "pred_team_a_goals",
    pred_b_col: str = "pred_team_b_goals",
) -> pd.DataFrame:
    required_test_cols = {"Id", "match_id", "team"}
    missing_test_cols = required_test_cols - set(test_row_df.columns)
    if missing_test_cols:
        raise KeyError(f"Kolom wajib pada test_row_df tidak ditemukan: {sorted(missing_test_cols)}")

    required_pred_cols = {"match_id", canonical_team_a_col, canonical_team_b_col, pred_a_col, pred_b_col}
    missing_pred_cols = required_pred_cols - set(pred_match_df.columns)
    if missing_pred_cols:
        raise KeyError(f"Kolom wajib pada pred_match_df tidak ditemukan: {sorted(missing_pred_cols)}")

    merged = test_row_df[["Id", "match_id", "team"]].copy().merge(
        pred_match_df[["match_id", canonical_team_a_col, canonical_team_b_col, pred_a_col, pred_b_col]],
        on="match_id",
        how="left",
        validate="m:1",
    )

    if merged[[pred_a_col, pred_b_col]].isna().any().any():
        raise ValueError("Masih ada prediksi match-level yang tidak berhasil dipetakan ke row-level.")

    team_is_a = merged["team"].astype(str) == merged[canonical_team_a_col].astype(str)
    team_is_b = merged["team"].astype(str) == merged[canonical_team_b_col].astype(str)

    unmatched_mask = ~(team_is_a | team_is_b)
    if unmatched_mask.any():
        bad_rows = merged.loc[unmatched_mask, ["Id", "match_id", "team", canonical_team_a_col, canonical_team_b_col]].head(10)
        raise ValueError(
            "Terdapat row test yang tidak bisa dicocokkan dengan team_a/team_b canonical. "
            f"Contoh:\n{bad_rows}"
        )

    merged["team_goals"] = np.where(team_is_a, merged[pred_a_col], merged[pred_b_col])
    merged["opp_goals"] = np.where(team_is_a, merged[pred_b_col], merged[pred_a_col])

    submission = merged[["Id", "team_goals", "opp_goals"]].copy()
    return submission

In [ ]:

dummy_pred_match_df = test_match[["match_id", "team_a", "team_b"]].copy()
dummy_pred_match_df["pred_team_a_goals"] = 1
dummy_pred_match_df["pred_team_b_goals"] = 1

dummy_submission_df = match_predictions_to_submission(
    test_row_df=test_df,
    pred_match_df=dummy_pred_match_df,
    canonical_team_a_col="team_a",
    canonical_team_b_col="team_b",
    pred_a_col="pred_team_a_goals",
    pred_b_col="pred_team_b_goals",
)

print("Dummy submission preview:")
display(dummy_submission_df.head())

In [ ]:

submission_check_df = pd.DataFrame(
    {
        "check": [
            "same_number_of_rows_as_sample_submission",
            "same_Id_order_as_sample_submission",
            "submission_columns_exact",
        ],
        "result": [
            len(dummy_submission_df) == len(sample_sub_df),
            dummy_submission_df["Id"].tolist() == sample_sub_df["Id"].tolist() if "Id" in sample_sub_df.columns else False,
            dummy_submission_df.columns.tolist() == ["Id", "team_goals", "opp_goals"],
        ],
    }
)
submission_check_df

Helper reverse mapping ini memastikan bahwa workflow match-level tetap **kompatibel dengan format submit kompetisi**. Jadi, kita bisa bekerja lebih aman di level pertandingan tanpa kehilangan kemampuan untuk menghasilkan submission row-level yang valid.

## 09. Implementasi Offline Evaluator AW-MAE

Section ini mengimplementasikan evaluator offline sesuai definisi kompetisi. Ini sangat penting karena metrik kompetisi **bukan MAE biasa**: ada penalti exact score, penalti outcome, penalti goal difference, multiplier jika outcome salah, dan pembobotan berdasarkan turnamen.

In [ ]:

EXACT_PENALTY = 0.30
OUTCOME_PENALTY = 0.25
GD_PENALTY = 0.15
WRONG_OUTCOME_MULTIPLIER = 1.50
NONLINEAR_POWER = 1.50


def _outcome(a: int, b: int) -> int:
    if a > b:
        return 1
    if a < b:
        return -1
    return 0


def get_tournament_weight(tournament: str) -> float:
    t = str(tournament).lower().strip()

    if "fifa world cup" in t or t == "world cup":
        return 2.00

    if "afc championship" in t or "afc asian cup" in t or "asian cup" in t:
        return 1.80

    if "friendly" in t:
        return 0.96

    return 1.20


def official_match_loss(
    y_team_true: int,
    y_opp_true: int,
    y_team_pred: int,
    y_opp_pred: int,
) -> float:
    y_team_true = int(y_team_true)
    y_opp_true = int(y_opp_true)
    y_team_pred = int(y_team_pred)
    y_opp_pred = int(y_opp_pred)

    mae = (abs(y_team_true - y_team_pred) + abs(y_opp_true - y_opp_pred)) / 2.0

    exact = int((y_team_true == y_team_pred) and (y_opp_true == y_opp_pred))
    outcome = int(_outcome(y_team_true, y_opp_true) == _outcome(y_team_pred, y_opp_pred))
    gd = int((y_team_true - y_opp_true) == (y_team_pred - y_opp_pred))

    penalty = (
        EXACT_PENALTY * (1 - exact)
        + OUTCOME_PENALTY * (1 - outcome)
        + GD_PENALTY * (1 - gd)
    )

    raw_loss = mae + penalty
    multiplier = 1.0 if outcome == 1 else WRONG_OUTCOME_MULTIPLIER
    loss = (raw_loss * multiplier) ** NONLINEAR_POWER
    return float(loss)


def awmae_score(
    y_team_true,
    y_opp_true,
    y_team_pred,
    y_opp_pred,
    tournaments,
) -> float:
    y_team_true = pd.to_numeric(pd.Series(y_team_true), errors="coerce").to_numpy()
    y_opp_true = pd.to_numeric(pd.Series(y_opp_true), errors="coerce").to_numpy()
    y_team_pred = pd.to_numeric(pd.Series(y_team_pred), errors="coerce").to_numpy()
    y_opp_pred = pd.to_numeric(pd.Series(y_opp_pred), errors="coerce").to_numpy()
    tournaments = pd.Series(tournaments).astype(str).to_numpy()

    n = len(y_team_true)
    lengths = [len(y_opp_true), len(y_team_pred), len(y_opp_pred), len(tournaments)]
    if any(length != n for length in lengths):
        raise ValueError("Semua input ke awmae_score harus memiliki panjang yang sama.")

    if np.isnan(y_team_true).any() or np.isnan(y_opp_true).any() or np.isnan(y_team_pred).any() or np.isnan(y_opp_pred).any():
        raise ValueError("Input numerik ke awmae_score mengandung NaN setelah konversi numerik.")

    losses = np.array(
        [
            official_match_loss(a, b, c, d)
            for a, b, c, d in zip(y_team_true, y_opp_true, y_team_pred, y_opp_pred)
        ],
        dtype=float,
    )
    weights = np.array([get_tournament_weight(t) for t in tournaments], dtype=float)

    if np.any(weights <= 0):
        raise ValueError("Semua tournament weight harus bernilai positif.")

    return float(np.average(losses, weights=weights))

In [ ]:

sanity_rows = []

# 1) perfect prediction -> 0
score_perfect = awmae_score(
    y_team_true=[2],
    y_opp_true=[1],
    y_team_pred=[2],
    y_opp_pred=[1],
    tournaments=["friendly"],
)
sanity_rows.append({"case": "perfect_prediction", "score": score_perfect})

# 2) near prediction but wrong outcome -> should be worse
score_wrong_outcome = awmae_score(
    y_team_true=[2],
    y_opp_true=[1],
    y_team_pred=[1],
    y_opp_pred=[2],
    tournaments=["friendly"],
)
sanity_rows.append({"case": "near_but_wrong_outcome", "score": score_wrong_outcome})

# 3) same outcome and same GD but not exact -> should still be penalized, but better than wrong outcome in many cases
score_same_outcome_same_gd_not_exact = awmae_score(
    y_team_true=[3],
    y_opp_true=[1],
    y_team_pred=[2],
    y_opp_pred=[0],
    tournaments=["friendly"],
)
sanity_rows.append({"case": "same_outcome_same_gd_not_exact", "score": score_same_outcome_same_gd_not_exact})

# 4) identical pair losses but placed on different tournament weights
score_high_error_on_world_cup = awmae_score(
    y_team_true=[3, 1],
    y_opp_true=[0, 1],
    y_team_pred=[0, 3],
    y_opp_pred=[3, 1],
    tournaments=["fifa world cup", "friendly"],
)
score_high_error_on_friendly = awmae_score(
    y_team_true=[3, 1],
    y_opp_true=[0, 1],
    y_team_pred=[0, 1],
    y_opp_pred=[3, 1],
    tournaments=["friendly", "fifa world cup"],
)
sanity_rows.append({"case": "high_error_on_world_cup", "score": score_high_error_on_world_cup})
sanity_rows.append({"case": "high_error_on_friendly", "score": score_high_error_on_friendly})

sanity_eval_df = pd.DataFrame(sanity_rows)
sanity_eval_df

In [ ]:

assert abs(score_perfect - 0.0) < 1e-12, "Perfect prediction seharusnya menghasilkan AW-MAE = 0."
assert score_wrong_outcome > score_same_outcome_same_gd_not_exact, "Wrong outcome seharusnya lebih buruk."
assert score_high_error_on_world_cup > score_high_error_on_friendly, "Error besar pada World Cup seharusnya memberi dampak average yang lebih besar."

print("Sanity check evaluator berhasil dilewati.")

Interpretasi dari evaluator ini penting untuk eksperimen lanjutan:

- metrik sangat sensitif terhadap **kesalahan outcome**,
- exact score tetap penting, tetapi salah menang/seri/kalah akan diberi hukuman tambahan,
- goal difference juga diperhitungkan,
- pembobotan turnamen membuat sebagian pertandingan lebih berpengaruh daripada yang lain.

Artinya, eksperimen berikutnya tidak cukup hanya mengoptimalkan error rata-rata biasa. Model dan decoding prediksi harus mempertimbangkan struktur skor pertandingan secara joint.

## 10. Temporal Validation Split yang Leakage-Safe

Karena test berada di masa setelah train, validasi offline yang realistis harus bersifat **temporal**, bukan random. Split juga harus dilakukan pada level pertandingan, bukan row, agar tidak terjadi leakage antar pasangan row dari match yang sama.

In [ ]:

def make_time_based_holdout(
    train_match: pd.DataFrame,
    valid_fraction: float = 0.2,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if "date" not in train_match.columns:
        raise KeyError("Kolom `date` wajib ada pada train_match untuk split temporal.")
    if train_match["date"].isna().any():
        raise ValueError("Kolom `date` pada train_match masih mengandung NaT. Bersihkan parsing date terlebih dahulu.")
    if not (0 < valid_fraction < 1):
        raise ValueError("valid_fraction harus berada di antara 0 dan 1.")

    ordered = train_match.sort_values(["date", "match_id"], kind="stable").reset_index(drop=True)
    n_total = len(ordered)
    n_valid = max(1, int(math.ceil(n_total * valid_fraction)))
    n_train = n_total - n_valid

    if n_train <= 0:
        raise ValueError("Ukuran validation terlalu besar; train split menjadi kosong.")

    train_fold = ordered.iloc[:n_train].copy().reset_index(drop=True)
    valid_fold = ordered.iloc[n_train:].copy().reset_index(drop=True)
    return train_fold, valid_fold

In [ ]:

train_match_sorted = train_match.sort_values(["date", "match_id"], kind="stable").reset_index(drop=True)
display(train_match_sorted[["match_id", "date", "team_a", "team_b"]].head(10))
display(train_match_sorted[["match_id", "date", "team_a", "team_b"]].tail(10))

In [ ]:

train_fold, valid_fold = make_time_based_holdout(train_match, valid_fraction=0.2)

split_summary_df = pd.DataFrame(
    {
        "split": ["train_fold", "valid_fold"],
        "n_matches": [len(train_fold), len(valid_fold)],
        "date_min": [train_fold["date"].min(), valid_fold["date"].min()],
        "date_max": [train_fold["date"].max(), valid_fold["date"].max()],
    }
)
split_summary_df

In [ ]:

leakage_check_df = pd.DataFrame(
    {
        "check": [
            "max_train_date_le_min_valid_date",
            "no_match_id_overlap",
            "train_fold_not_empty",
            "valid_fold_not_empty",
        ],
        "result": [
            train_fold["date"].max() <= valid_fold["date"].min(),
            set(train_fold["match_id"]).isdisjoint(set(valid_fold["match_id"])),
            len(train_fold) > 0,
            len(valid_fold) > 0,
        ],
    }
)
leakage_check_df

Split temporal wajib pada kompetisi ini karena:

- train dan test dipisahkan oleh waktu,
- random split akan memberi validasi yang terlalu optimistis,
- leakage bisa muncul jika satu match masuk ke train dan valid secara terpisah pada level row,
- holdout temporal di level match jauh lebih dekat dengan skenario inference yang sebenarnya.

## 11. Audit Distribusi Dasar yang Relevan untuk Eksperimen Selanjutnya

Section ini bukan EDA yang terlalu luas, tetapi audit distribusi yang memang relevan untuk merancang pipeline lanjutan.

In [ ]:

if "gender" in train_df.columns:
    print("Distribusi gender pada train:")
    display(train_df["gender"].value_counts(dropna=False).rename_axis("gender").reset_index(name="count"))

if "neutral" in train_df.columns:
    print("\nDistribusi neutral pada train:")
    display(train_df["neutral"].value_counts(dropna=False).rename_axis("neutral").reset_index(name="count"))

if "tournament" in train_df.columns:
    print("\nTop tournament pada train:")
    display(train_df["tournament"].value_counts(dropna=False).head(20).rename_axis("tournament").reset_index(name="count"))

confederation_candidate_cols = [col for col in train_df.columns if "confederation" in col.lower()]
if confederation_candidate_cols:
    conf_col = confederation_candidate_cols[0]
    print(f"\nDistribusi confederation pada train (menggunakan kolom `{conf_col}`):")
    display(train_df[conf_col].value_counts(dropna=False).head(20).rename_axis(conf_col).reset_index(name="count"))
else:
    print("\nTidak ada kolom confederation yang ditemukan pada train.")

In [ ]:

score_dist_rows = []

for col in ["team_goals", "opp_goals"]:
    if col in train_df.columns:
        vc = train_df[col].value_counts(dropna=False).sort_index()
        tmp = vc.rename_axis("score").reset_index(name="count")
        tmp["target_col"] = col
        score_dist_rows.append(tmp)

if score_dist_rows:
    score_dist_df = pd.concat(score_dist_rows, ignore_index=True)
    display(score_dist_df.head(50))
else:
    print("Kolom target skor tidak ditemukan pada train.")

In [ ]:

if {"team_goals", "opp_goals"}.issubset(train_df.columns):
    scoreline_df = train_df[["team_goals", "opp_goals"]].copy()
    scoreline_df["scoreline"] = scoreline_df["team_goals"].astype(str) + "-" + scoreline_df["opp_goals"].astype(str)
    top_scoreline_df = scoreline_df["scoreline"].value_counts().head(20).rename_axis("scoreline").reset_index(name="count")
    display(top_scoreline_df)
else:
    print("Kolom target belum lengkap untuk menghitung exact scoreline.")

In [ ]:

if "team" in train_df.columns and "team" in test_df.columns:
    train_teams = set(train_df["team"].dropna().astype(str).unique())
    test_teams = set(test_df["team"].dropna().astype(str).unique())
    unseen_teams = sorted(test_teams - train_teams)

    unseen_team_summary_df = pd.DataFrame(
        {
            "n_unique_train_teams": [len(train_teams)],
            "n_unique_test_teams": [len(test_teams)],
            "n_unseen_test_teams": [len(unseen_teams)],
            "sample_unseen_test_teams": [unseen_teams[:20]],
        }
    )
    display(unseen_team_summary_df)
else:
    print("Kolom `team` tidak tersedia pada train/test untuk audit unseen teams.")

In [ ]:

descriptive_shift_tables = {}

for col in ["gender", "tournament"]:
    if col in train_df.columns and col in test_df.columns:
        shift_df = (
            pd.concat(
                [
                    train_df[col].value_counts(normalize=True, dropna=False).rename("train_prop"),
                    test_df[col].value_counts(normalize=True, dropna=False).rename("test_prop"),
                ],
                axis=1,
            )
            .fillna(0.0)
            .sort_values("train_prop", ascending=False)
            .reset_index()
            .rename(columns={"index": col})
        )
        descriptive_shift_tables[col] = shift_df

for col, table in descriptive_shift_tables.items():
    print(f"Perbandingan proporsi deskriptif train vs test untuk kolom `{col}`:")
    display(table.head(30))

Insight yang biasanya ingin dibawa ke eksperimen berikutnya dari section ini:

- distribusi train dan test bisa berbeda secara deskriptif,
- ada kemungkinan muncul **unseen teams** pada test,
- pola scoreline dominan dapat menjadi baseline awal,
- distribusi gender dan tournament penting untuk memahami kemungkinan distribution shift.

## 12. Shared-Features Feasibility Audit

Section ini memisahkan dengan tegas fitur yang **feasible** dipakai saat inference test dari fitur yang hanya tersedia di train. Tujuannya agar eksperimen berikutnya tidak membangun validasi yang terlalu optimistis.

In [ ]:

raw_excluded_cols = {
    "Id",
    "match_id",
    "team_goals",
    "opp_goals",
}

shared_feature_cols = sorted((set(train_df.columns) & set(test_df.columns)) - raw_excluded_cols)
train_only_feature_cols = sorted((set(train_df.columns) - set(test_df.columns)) - {"team_goals", "opp_goals", "Id", "match_id"})

print("Shared feature columns:")
for idx, col in enumerate(shared_feature_cols, start=1):
    print(f"{idx:03d}. {col}")

print("\nTrain-only feature columns:")
for idx, col in enumerate(train_only_feature_cols, start=1):
    print(f"{idx:03d}. {col}")

Interpretasi feasibility audit:

- baseline awal yang sehat sebaiknya dibangun dari **shared features**,
- fitur train-only perlu diperlakukan sangat hati-hati karena tidak tersedia saat inference test,
- validasi yang memakai fitur train-only tanpa strategi inference yang jelas dapat menyesatkan dan tampak terlalu bagus secara offline.

## 13. Baseline Sanity Check Sederhana

Baseline pada section ini **bukan** baseline final kompetisi. Tujuannya hanya memastikan bahwa:

- split temporal berjalan,
- evaluator AW-MAE berjalan,
- pipeline match-level ke evaluasi berjalan tanpa error,
- dan kita bisa melihat contoh prediksi vs aktual.

In [ ]:

if {"team_a_goals", "team_b_goals", "tournament"}.issubset(train_fold.columns) and {"team_a_goals", "team_b_goals", "tournament"}.issubset(valid_fold.columns):
    train_scoreline_counts = (
        train_fold[["team_a_goals", "team_b_goals"]]
        .value_counts()
        .reset_index(name="count")
        .sort_values(["count", "team_a_goals", "team_b_goals"], ascending=[False, True, True])
        .reset_index(drop=True)
    )
    most_common_team_a_goals = int(train_scoreline_counts.loc[0, "team_a_goals"])
    most_common_team_b_goals = int(train_scoreline_counts.loc[0, "team_b_goals"])

    valid_pred_df = valid_fold[["match_id", "date", "tournament", "team_a", "team_b", "team_a_goals", "team_b_goals"]].copy()
    valid_pred_df["pred_team_a_goals"] = most_common_team_a_goals
    valid_pred_df["pred_team_b_goals"] = most_common_team_b_goals

    baseline_awmae = awmae_score(
        y_team_true=valid_pred_df["team_a_goals"],
        y_opp_true=valid_pred_df["team_b_goals"],
        y_team_pred=valid_pred_df["pred_team_a_goals"],
        y_opp_pred=valid_pred_df["pred_team_b_goals"],
        tournaments=valid_pred_df["tournament"],
    )

    baseline_result_df = pd.DataFrame(
        {
            "baseline_name": ["most_common_match_level_scoreline"],
            "pred_team_a_goals": [most_common_team_a_goals],
            "pred_team_b_goals": [most_common_team_b_goals],
            "valid_awmae": [baseline_awmae],
            "n_valid_matches": [len(valid_pred_df)],
        }
    )
    display(baseline_result_df)

    print("Contoh prediksi vs aktual pada validation fold:")
    display(valid_pred_df.head(20))
else:
    print("Kolom yang dibutuhkan untuk baseline sanity check belum lengkap.")

Baseline ini hanya dipakai sebagai **sanity check**, bukan sebagai acuan performa akhir. Kalau section ini berhasil jalan, berarti komponen-komponen inti notebook sudah terhubung dengan baik:

- holdout temporal berhasil,
- target match-level berhasil,
- evaluator AW-MAE berhasil,
- dan prediksi bisa dibandingkan dengan aktual tanpa error.

## 14. Ringkasan Hasil Eksperimen EXP 00

Temuan utama dari EXP 00 yang diharapkan setelah notebook dijalankan:

- file train, test, sample submission, dan metadata berhasil dibaca dari `../data/`,
- struktur dua-row-per-match berhasil diaudit,
- representasi **match-level canonical** berhasil dibangun,
- helper reverse mapping ke format submission berhasil dibuat,
- evaluator **offline AW-MAE** berhasil diimplementasikan,
- pembobotan turnamen berhasil ditanamkan ke evaluator,
- temporal validation holdout berhasil disiapkan tanpa overlap match,
- gap fitur train vs test berhasil diidentifikasi,
- baseline sanity check berhasil dijalankan untuk memastikan seluruh pipeline bekerja.

## 15. Next Step ke EXP 01

Eksperimen berikutnya (EXP 01) sebaiknya mulai fokus pada:

- baseline model awal yang **feasible terhadap test**,
- penggunaan **shared features** terlebih dahulu,
- representasi **match-level** yang sama,
- validasi temporal yang sama,
- evaluasi dengan **offline AW-MAE** yang sama,
- dan mulai membangun fondasi modeling yang kompatibel dengan format submit kompetisi.